# Predicting the Next Basket as a Joint Object: Which Products, and How Many

## What the project predicts

Every time a customer opens a grocery app, most of what they buy is not a discovery. It is a refill. Milk, bananas, the same brand of coffee, the detergent they always get. The original question this dataset invites is *which* of a user's past products return in the next order. This project asks a tighter, jointly specified question: not only which products recur, but how many of them do.

Two prediction heads sit on the same event, the user's next basket. The first is the per product reorder classifier. For every pair $(u, p)$ where user $u$ has purchased product $p$ before,

$$y_{u,p} = \begin{cases} 1 & \text{if } p \text{ appears in } u\text{'s next order} \\ 0 & \text{otherwise.} \end{cases}$$

The second head is a count regressor. Let $B_u^{\text{repeat}}$ be the number of previously bought products that reappear in user $u$'s next order:

$$B_u^{\text{repeat}} = \sum_{p \in \text{hist}(u)} \mathbf{1}[y_{u,p} = 1].$$

The first head is binary classification over roughly five million candidate pairs from 131,000 users. The second is count regression, one target per user. The project's contribution is not running both, it is the identity that binds them and the decision rule that identity unlocks.

## The central thesis: how much does structure beat the floor

Reorder behaviour is strongly autocorrelated. What a user buys often, they tend to buy again. The single feature that captures this first order effect is the user product purchase frequency $f_{u,p}$, the fraction of a user's past orders containing product $p$:

$$f_{u,p} = \frac{\#\{\text{orders of } u \text{ containing } p\}}{N_u}.$$

Thresholding $f_{u,p}$ directly gives a baseline classifier, the rule *predict what you buy most often*. It is the most important single feature on this dataset, which is exactly why it is the right floor to measure against. Reporting it first means every later number reads as an improvement over a stated reference rather than a score appearing from nowhere.

Whether structured models actually beat that floor is an open question, not a settled one, and answering it is the headline of the project:

$$\Delta_{\text{clf}} = F_1^{\text{best model}} - F_1^{\,f_{u,p}\text{ heuristic}}.$$

There is good reason to expect the floor leaves signal on the table, because frequency is a single all time ratio and there are specific things it provably cannot see. It cannot distinguish a product bought in the last three orders straight but only 20% of the time overall, a habit forming, from one bought 20% of the time but not in the last ten orders, a habit fading. Identical frequency, opposite trajectory; only recency and streak separate them. It also cannot use product base rates: milk and a seasonal item with identical $f_{u,p}$ have different true reorder odds, and only the product features carry that. Whether those blind spots hold enough signal to move the metric is precisely what $\Delta_{\text{clf}}$ measures.

The framing cuts both ways and both directions are honest results. A large $\Delta_{\text{clf}}$ means the blind spots mattered and the engineering bought real signal beyond raw recurrence. A small one means reorder behaviour really is dominated by a single autocorrelation term and the residual structure is close to irreducible. The project does not promise either outcome. It builds the instrument that returns one.

## The identity that links the two heads

The two heads are not independent facts about a user. By linearity of expectation, the expected repeat basket size is the sum of the classifier's own probabilities:

$$\mathbb{E}\big[B_u^{\text{repeat}}\big] = \sum_{p \in \text{hist}(u)} P(y_{u,p}=1).$$

The classifier therefore predicts basket size implicitly. The count regressor, trained on user level features, supplies a second independent estimate of the same quantity. Where the two agree the prediction is trustworthy; where they diverge, the gap is a diagnostic that points straight at miscalibration.

Worked instance: a user has 30 products in their history and the classifier's 30 probabilities sum to $6.2$, so the classifier expects about $6.2$ repeat items. The count regressor, looking only at user level features ($\bar B_u^{\text{repeat}}$, recency, reorder ratio), independently predicts $7$. That gap of $0.8$ is information. If the regressor is right, the classifier's probabilities are collectively underconfident, exactly the miscalibration the Brier and isotonic step exists to catch, now with a second witness.

## The third decision rule

The original project turns probabilities into a predicted set with a threshold: include $p$ if $P(y_{u,p}=1) > \tau$. The count head offers a different rule entirely.

There is a structural fact about per user $F_1$. Sort a user's candidates by probability, $p_1 \ge p_2 \ge \dots \ge p_m$. The $F_1$ maximising prediction set is always a prefix: take the top $k$ for some $k$ and never skip a higher probability item to reach a lower one. So the whole decision collapses to choosing the cardinality $k$, and $k$ is precisely what a basket size model predicts. Set $k = \text{round}(\hat B_u^{\text{repeat}})$ and take that user's top $k$ products.

That gives three decision rules to compare on the same metric: the global threshold $\tau$, the per user threshold $\tau_u$, and predicted cardinality. The user above with predicted count $7$ gets their seven highest probability products with no threshold involved. If thresholding at $\tau = 0.2$ would have returned only five, the cardinality rule recovers the two it was dropping. The size model stops being a side result and feeds the headline metric directly.

## Two thesis margins, not one

The regressor gets its own floor and its own open question, every user predicting their own historical mean:

$$\Delta_{\text{reg}} = \text{error}^{\,\bar B_u^{\text{repeat}} \text{ baseline}} - \text{error}^{\text{count model}}.$$

Whether features beat the user's own average is as genuinely open as whether models beat the frequency heuristic. A small $\Delta_{\text{reg}}$ states that repeat basket size is essentially each user's own average; a large one states that behavioural features sharpen it. Both heads report the margin over a stated baseline rather than a score from nowhere, and neither margin is assumed in advance.

## Two ceilings that exist before any model runs

The classifier inherits the recall ceiling. A product bought for the first time in the final order is never a candidate, so with $R_{\text{new}}$ the fraction of final order items that are first time purchases,

$$\text{recall}_{\max} = 1 - R_{\text{new}}.$$

The regressor gets its own ceiling, the irreducible within user variance. A user whose repeat basket size genuinely fluctuates cannot be predicted below their own noise, so the error floor is

$$\text{RMSE}_{\min} \approx \sqrt{\mathbb{E}_u\!\big[\sigma^2_{B^{\text{repeat}}, u}\big]},$$

the average within user standard deviation, measured before any model runs the same way $R_{\text{new}}$ is. Example: a user averages 7 repeat items with a within user standard deviation of 2. No model beats RMSE 2 for that user, because that swing is real behaviour. If the average over users is 2.3, then $\text{RMSE}_{\min} \approx 2.3$ is the ceiling, and a model at RMSE 2.5 has captured nearly all the predictable structure.

The two ceilings even connect. Predicting total basket size alongside repeat count yields a per user novelty estimate $\hat R_{\text{new}, u} = (\hat B_u^{\text{total}} - \hat B_u^{\text{repeat}}) / \hat B_u^{\text{total}}$, turning the single global $R_{\text{new}}$ into a quantity forecast per user.

## How the project is structured

Four phases, each closing with versioned commits, research narrative kept separate from reusable code.

**Phase 1, Foundation.** Load all six raw tables, convert to parquet with int32 and float32 downcasting, run EDA where every finding ties to a downstream decision. The phase confirms with a Mann Whitney U test that $f_{u,p}$ separates the $y=1$ and $y=0$ populations, characterises the repeat basket size distribution the count head will model, and measures both ceilings.

**Phase 2, Feature Engineering.** Build the user fingerprint $\phi_u$, the product profile $\phi_p$, and the interaction features $\phi_{u,p}$ that carry the classifier's signal, plus the regression target $B_u^{\text{repeat}}$. The classifier reads all three families; the count regressor reads $\phi_u$.

**Phase 3, Modelling.** The classifier lineup spans an inductive bias axis: the frequency heuristic floor, logistic regression as the linear floor, LightGBM as the primary model, and an MLP as a representation test of whether the hand engineered interaction features already captured what a network would learn. The count head spans its own smaller axis, all non neural: the mean baseline, a Poisson GLM as the linear floor, and a negative binomial GBDT, with the negative binomial choice decided by an overdispersion test rather than assumed. No neural net on the regression side, because its target has no interaction features for a network to discover, so an MLP there would test nothing.

**Phase 4, Evaluation and Write up.** Calibrate the classifier probabilities and check the count head's prediction intervals, the two heads' matching calibration steps. Derive and verify $\tau^* = F_1^*/2$, then run the three way decision rule comparison (global threshold, per user threshold, predicted cardinality) as the section where the heads meet, alongside the reconciliation diagnostic between the probability sum and the regressor. Close with the LightGBM versus MLP comparison under a Nadeau Bengio corrected test and the feature ablation.

## What success looks like

Success is not a high score on either head. Both ceilings make that unattainable and a raw number misleading without context, and the two margins are genuinely open questions rather than promised results. Success is a notebook where each head states its floor and measures its margin over it without assuming the answer, both ceilings are quantified before modelling, the identity linking the heads is used as a live calibration check, and the cardinality rule shows whether predicting how many sharpens predicting which. The headline is the pair $(\Delta_{\text{clf}}, \Delta_{\text{reg}})$ read against $1 - R_{\text{new}}$ and $\text{RMSE}_{\min}$, so the reader always knows both how far the modelling moved each score and how far it could possibly have moved.